In [13]:
# ============================================================
# TASK 4 - CUMULATIVE APP INSTALL TRENDS
# ============================================================

import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import webbrowser

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv("Play Store Data.csv")

print("Dataset loaded successfully!")
print("Original shape:", df.shape)


# ------------------------------------------------------------
# 2. CLEAN DATA
# ------------------------------------------------------------

# Rating
df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors="coerce"
)

# Reviews
df["Reviews"] = pd.to_numeric(
    df["Reviews"],
    errors="coerce"
)

# Installs
df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

df["Installs"] = pd.to_numeric(
    df["Installs"],
    errors="coerce"
)

# Size
def convert_size(size):

    size = str(size).strip()

    if size in ["nan", "NaN"]:
        return None

    if size.endswith("M"):
        return float(size[:-1])

    if size.endswith("m"):
        return float(size[:-1])

    if size.endswith("K"):
        return float(size[:-1]) / 1024

    if size.endswith("k"):
        return float(size[:-1]) / 1024

    return None


df["Size_MB"] = df["Size"].apply(
    convert_size
)

# Last Updated
df["Last Updated"] = pd.to_datetime(
    df["Last Updated"],
    errors="coerce"
)

print("Data cleaning completed!")


# ------------------------------------------------------------
# 3. APPLY TASK 4 FILTERS
# ------------------------------------------------------------

task4 = df[
    # Average rating at least 4.2
    (df["Rating"] >= 4.2)

    # App name must NOT contain any numbers
    & (
        ~df["App"]
        .astype(str)
        .str.contains(
            r"\d",
            regex=True,
            na=False
        )
    )

    # Category must start with T or P
    & (
        df["Category"]
        .astype(str)
        .str.startswith(
            ("T", "P"),
            na=False
        )
    )

    # Reviews greater than 1,000
    & (df["Reviews"] > 1000)

    # App size between 20 MB and 80 MB
    & (df["Size_MB"].between(20, 80))
].copy()

print()
print("Task 4 filtering completed!")
print("Filtered rows:", len(task4))
print(
    "Categories:",
    sorted(task4["Category"].unique())
)


# ------------------------------------------------------------
# 4. CATEGORY TRANSLATIONS
# ------------------------------------------------------------

category_translation = {

    # Travel & Local → French
    "TRAVEL_AND_LOCAL":
        "Voyage et local",

    # Productivity → Spanish
    "PRODUCTIVITY":
        "Productividad",

    # Photography → Japanese
    "PHOTOGRAPHY":
        "写真"
}

task4["Display_Category"] = (
    task4["Category"]
    .map(category_translation)
    .fillna(task4["Category"])
)

print()
print("Category translations added!")

print(
    task4[
        ["Category", "Display_Category"]
    ].drop_duplicates()
)


# ------------------------------------------------------------
# 5. CREATE MONTH
# ------------------------------------------------------------

task4["Month"] = (
    task4["Last Updated"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# ------------------------------------------------------------
# 6. MONTHLY INSTALLS
# ------------------------------------------------------------

monthly = (
    task4
    .groupby(
        [
            "Category",
            "Display_Category",
            "Month"
        ]
    )["Installs"]
    .sum()
    .reset_index(
        name="Monthly_Installs"
    )
)

monthly = monthly.sort_values(
    [
        "Category",
        "Month"
    ]
)


# ------------------------------------------------------------
# 7. CUMULATIVE INSTALLS
# ------------------------------------------------------------

monthly["Cumulative_Installs"] = (
    monthly
    .groupby("Category")[
        "Monthly_Installs"
    ]
    .cumsum()
)


# ------------------------------------------------------------
# 8. MONTH-OVER-MONTH GROWTH
# ------------------------------------------------------------

monthly["MoM_Growth"] = (
    monthly
    .groupby("Category")[
        "Monthly_Installs"
    ]
    .pct_change()
    * 100
)

monthly["Significant_Growth"] = (
    monthly["MoM_Growth"] > 25
)

print()
print("MoM growth calculation completed!")

print(
    "Months with growth above 25%:",
    monthly["Significant_Growth"].sum()
)


# ------------------------------------------------------------
# 9. PREPARE STACKED AREA DATA
# ------------------------------------------------------------

area_data = (
    monthly
    .pivot_table(
        index="Month",
        columns="Display_Category",
        values="Cumulative_Installs",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

print()
print("Area chart data prepared!")

print(
    "Number of months:",
    len(area_data)
)

print(
    "Categories:",
    list(area_data.columns[1:])
)


# ------------------------------------------------------------
# 10. CREATE STACKED AREA CHART
# ------------------------------------------------------------

fig = go.Figure()

categories = list(
    area_data.columns[1:]
)

for category in categories:

    fig.add_trace(
        go.Scatter(
            x=area_data["Month"],
            y=area_data[category],

            mode="lines",

            name=category,

            stackgroup="one",

            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Month: %{x|%b %Y}<br>"
                "Cumulative Installs: "
                "%{y:,.0f}"
                "<extra></extra>"
            )
        )
    )


# ------------------------------------------------------------
# 11. HIGHLIGHT >25% GROWTH MONTHS
# ------------------------------------------------------------

highlight_months = (
    monthly.loc[
        monthly["Significant_Growth"],
        "Month"
    ]
    .dropna()
    .unique()
)

for month in highlight_months:

    month = pd.Timestamp(month)

    fig.add_vrect(

        x0=month,

        x1=(
            month
            + pd.offsets.MonthBegin(1)
        ),

        fillcolor="red",

        opacity=0.12,

        line_width=0,

        layer="below"
    )


# ------------------------------------------------------------
# 12. CHART DESIGN
# ------------------------------------------------------------

fig.update_layout(

    title=(
        "Cumulative App Install "
        "Trends by Category"
    ),

    xaxis_title="Month",

    yaxis_title="Cumulative Installs",

    hovermode="x unified",

    template="plotly_white",

    height=650,

    legend_title="App Category",

    margin=dict(
        l=60,
        r=40,
        t=80,
        b=60
    )
)


# ------------------------------------------------------------
# 13. CONVERT CHART TO HTML
# ------------------------------------------------------------

plot_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True
)


# ------------------------------------------------------------
# 14. CREATE FINAL DASHBOARD HTML
# ------------------------------------------------------------

dashboard_html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>
Cumulative App Install Trends
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 0;
    background: #f5f5f5;
}}

.header {{
    background: #222;
    color: white;
    padding: 20px;
    text-align: center;
}}

.container {{
    width: 95%;
    margin: 20px auto;
    background: white;
    padding: 20px;
    border-radius: 10px;
}}

#dashboard {{
    display: none;
}}

#outside-message {{
    display: none;
    text-align: center;
    padding: 100px 20px;
    font-size: 22px;
    color: #555;
}}

.info {{
    margin-bottom: 20px;
    padding: 15px;
    background: #f0f0f0;
    border-radius: 8px;
    line-height: 1.6;
}}

</style>

</head>


<body>


<div class="header">

<h1>
Cumulative App Install Trends
</h1>

<p>
Stacked Area Chart by App Category
</p>

</div>


<div class="container">


<div id="dashboard">


<div class="info">

<b>Filters applied:</b>

<br>

Rating ≥ 4.2 |
App name contains no numbers |
Category starts with T or P |
Reviews &gt; 1,000 |
App size 20–80 MB

<br><br>

<b>Category translations:</b>

Travel & Local → Voyage et local |

Productivity → Productividad |

Photography → 写真

<br><br>

<b>Growth highlight:</b>

Months where installs increased by more than
25% month-over-month are highlighted.

</div>


{plot_html}


</div>


<div id="outside-message">

<h2>
Dashboard Currently Unavailable
</h2>

<p>

This dashboard is available only between

<b>
4:00 PM and 6:00 PM IST
</b>

</p>

</div>


</div>


<script>


function checkTime() {{

    const now = new Date();

    const istTime =
        new Intl.DateTimeFormat(
            "en-US",
            {{
                timeZone: "Asia/Kolkata",
                hour: "2-digit",
                minute: "2-digit",
                hour12: false
            }}
        ).formatToParts(now);


    let hour = parseInt(

        istTime.find(
            p => p.type === "hour"
        ).value

    );


    const minute = parseInt(

        istTime.find(
            p => p.type === "minute"
        ).value

    );


    const currentMinutes =
        hour * 60 + minute;


    const startTime =
        16 * 60;


    const endTime =
        18 * 60;


    const dashboard =
        document.getElementById(
            "dashboard"
        );


    const message =
        document.getElementById(
            "outside-message"
        );


    if (
        currentMinutes >= startTime &&
        currentMinutes < endTime
    ) {{

        dashboard.style.display =
            "block";

        message.style.display =
            "none";

    }}

    else {{

        dashboard.style.display =
            "none";

        message.style.display =
            "block";

    }}

}}


// Check immediately
checkTime();


// Check every minute
setInterval(
    checkTime,
    60000
);


</script>


</body>

</html>
"""


# ------------------------------------------------------------
# 15. SAVE HTML FILE
# ------------------------------------------------------------

dashboard_file = Path(
    "dashboard_task4.html"
)

dashboard_file.write_text(
    dashboard_html,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 16. OPEN HTML AUTOMATICALLY
# ------------------------------------------------------------

webbrowser.open(
    dashboard_file
    .resolve()
    .as_uri()
)


# ------------------------------------------------------------
# 17. SHOW CHART IN NOTEBOOK
# ------------------------------------------------------------

fig.show()


# ------------------------------------------------------------
# 18. FINAL OUTPUT
# ------------------------------------------------------------

print()
print("=" * 60)

print(
    "TASK 4 COMPLETED SUCCESSFULLY!"
)

print("=" * 60)

print(
    "Filtered rows:",
    len(task4)
)

print(
    "Monthly records:",
    len(monthly)
)

print(
    "Categories:",
    categories
)

print(
    "Highlighted growth months:",
    len(highlight_months)
)

print(
    "Dashboard file:",
    dashboard_file.resolve()
)

print(
    "Dashboard available:",
    "4:00 PM - 6:00 PM IST"
)

print("=" * 60)

Dataset loaded successfully!
Original shape: (10841, 13)
Data cleaning completed!

Task 4 filtering completed!
Filtered rows: 139
Categories: ['PARENTING', 'PERSONALIZATION', 'PHOTOGRAPHY', 'PRODUCTIVITY', 'TOOLS', 'TRAVEL_AND_LOCAL']

Category translations added!
              Category Display_Category
2802       PHOTOGRAPHY               写真
3105  TRAVEL_AND_LOCAL  Voyage et local
3321             TOOLS            TOOLS
3363   PERSONALIZATION  PERSONALIZATION
3453      PRODUCTIVITY    Productividad
3607         PARENTING        PARENTING

MoM growth calculation completed!
Months with growth above 25%: 19

Area chart data prepared!
Number of months: 18
Categories: ['PARENTING', 'PERSONALIZATION', 'Productividad', 'TOOLS', 'Voyage et local', '写真']



TASK 4 COMPLETED SUCCESSFULLY!
Filtered rows: 139
Monthly records: 37
Categories: ['PARENTING', 'PERSONALIZATION', 'Productividad', 'TOOLS', 'Voyage et local', '写真']
Highlighted growth months: 9
Dashboard file: C:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task4.html
Dashboard available: 4:00 PM - 6:00 PM IST
